# Ch 20. 지도 미세조정 — 해설

> 이 노트북은 다섯 연습문제의 재현 가능한 해설을 제공한다. 모든 출력은 비워 두었으며 코드 셀은 위에서 아래로 독립적인 CPU 환경에서 실행된다.


## 문제 1

**문제**: 손실 마스킹을 켜고 끄고 각각 학습 후 응답 품질을 비교하라.

### 해설

응답 전용 마스킹은 프롬프트 토큰의 label을 $-100$으로 두어 손실에서 제외한다. 비교는 동일 초기 모델과 응답 토큰 예산에서 응답 NLL 및 고정 평가 루브릭으로 한다. 아래는 마스킹된 평균이 응답 위치만 포함함을 검증한다.


In [ ]:
import numpy as np

# Train one-token response classifiers with and without prompt-loss masking.
rng = np.random.default_rng(2001)
samples, features = 400, 6
X = rng.normal(size=(samples,features)); response = (X[:,0]+X[:,1]>0).astype(int); prompt = rng.integers(0,2,size=samples)
results={}
for masked in (False, True):
    w=np.zeros(features); lr=0.4
    for _ in range(100):
        logits=X@w; prob=1/(1+np.exp(-logits))
        target=response if masked else (response+prompt)/2
        w -= lr * X.T@(prob-target)/samples
    response_nll=-np.mean(response*np.log(prob+1e-9)+(1-response)*np.log(1-prob+1e-9))
    results[masked]={"response_nll":float(response_nll),"accuracy":float(np.mean((prob>.5)==response))}
assert results[True]["response_nll"] < results[False]["response_nll"] and results[True]["accuracy"] > results[False]["accuracy"]
print({("masked" if k else "unmasked"): {m:round(v,4) for m,v in r.items()} for k,r in results.items()})


## 문제 2

**문제**: 학습률 1e-2, 1e-3, 1e-4로 SFT하고 비교하라.

### 해설

SFT의 큰 학습률은 사전학습 표현을 급격히 훼손할 수 있다. 동일 스텝에서 train/validation 응답 손실, 그래디언트 노름, 사전학습 검증 손실의 변화를 함께 추적해 과적합과 catastrophic forgetting을 구분한다.


In [ ]:
import numpy as np

rng=np.random.default_rng(2002); X=rng.normal(size=(300,5)); true_w=rng.normal(size=5); y=X@true_w
pretrained=true_w+rng.normal(scale=.2,size=5); results={}
for lr in (1e-2,1e-3,1e-4):
    w=pretrained.copy(); curve=[]
    for _ in range(80):
        error=X@w-y; curve.append(float(np.mean(error**2))); w-=lr*2*X.T@error/len(X)
    results[lr]={"validation_mse":curve[-1],"distance_from_pretrained":float(np.linalg.norm(w-pretrained))}
assert results[1e-2]["validation_mse"] < results[1e-3]["validation_mse"] < results[1e-4]["validation_mse"]
print({str(k): {m:round(v,6) for m,v in r.items()} for k,r in results.items()})


## 문제 3

**문제**: instruction 데이터 5, 50, 500개로 늘려가며 SFT 효과를 비교하라.

### 해설

데이터 수만 바꾸되 업데이트 토큰 예산을 고정하면 데이터 다양성 효과를 분리할 수 있다. 5개에서는 반복 과적합, 500개에서는 더 넓은 일반화를 예상하지만 이는 가설이며 고정된 미학습 지시 세트에서 검증해야 한다.


In [ ]:
import numpy as np

# More diverse instructions cover more latent directions; evaluate on a held-out fixed set.
rng=np.random.default_rng(2003); pool=rng.normal(size=(500,12)); true_w=rng.normal(size=12); targets=pool@true_w
test=rng.normal(size=(300,12)); test_y=test@true_w; results={}
for n in (5,50,500):
    w,*_=np.linalg.lstsq(pool[:n],targets[:n],rcond=None)
    results[n]=float(np.mean((test@w-test_y)**2))
assert results[500] < 1e-20 and results[50] < results[5]
print({n:{"heldout_mse":f"{mse:.3e}"} for n,mse in results.items()})


## 문제 4

**문제**: ChatML 포맷으로 SFT 데이터를 변환하라.

### 해설

ChatML은 각 메시지를 역할 토큰과 내용, 종료 토큰으로 직렬화한다. 토크나이저의 실제 특수 토큰 정의를 사용해야 하며 문자열을 임의로 추정하지 않는다. 아래 변환은 구조 보존과 결정성을 검증한다.


In [ ]:
messages=[{"role":"system","content":"Be concise."},{"role":"user","content":"Add 2 and 3."},{"role":"assistant","content":"5"}]
def to_chatml(items):
    return "".join(f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>\n" for m in items)
serialized=to_chatml(messages)
for message in messages:
    assert f"<|im_start|>{message['role']}\n{message['content']}<|im_end|>" in serialized
assert serialized == to_chatml(messages)
print(serialized)


## 문제 5

**문제**: 사전학습 모델 vs SFT 모델의 응답을 비교하고 차이를 설명하라.

### 해설

사전학습 모델은 다음 토큰 분포를 학습했지만 지시-응답 경계를 따르도록 직접 최적화되지 않았다. SFT는 응답 형식과 종료 조건을 강화한다. 사실성 향상은 자동 보장이 아니므로 블라인드 루브릭으로 유용성·정확성·형식을 각각 평가한다.


In [ ]:
import numpy as np

# Blind rubric on held-out toy instructions: format compliance and task correctness are independent checks.
instructions=[("add",2,3),("multiply",3,4),("add",5,8),("multiply",2,7)]
def pretrained_response(item): return str(item[1])
def sft_response(item): return str(item[1]+item[2] if item[0]=="add" else item[1]*item[2])
expected=["5","12","13","14"]
scores={}
for name,model in (("pretrained",pretrained_response),("sft",sft_response)):
    responses=[model(x) for x in instructions]
    scores[name]={"exact_accuracy":float(np.mean(np.array(responses)==expected)),"numeric_format":float(np.mean([r.isdigit() for r in responses]))}
assert scores["sft"]["exact_accuracy"] > scores["pretrained"]["exact_accuracy"]
print(scores)
